# Step 6. 하이퍼파라미터 튜닝 — 기본값을 넘어 체계적 최적화

**목표**: Step 3에서 손으로 정한 기본 파라미터를 `RandomizedSearchCV`로 체계적으로 탐색해 RandomForest·XGBoost의 성능을 끌어올린다.

**누수 방지 원칙(동일)**
> 탐색은 **train 교차검증 안에서만** 한다. `SMOTE → 분류기`를 한 파이프라인으로 묶어 RandomizedSearchCV를 돌리면, SMOTE가 각 fold의 학습 부분에만 적용된다. test는 마지막 평가에만 쓴다.

**평가 지표**: 탐색 스코어는 **F1**(Recall·Precision 균형). 최종 비교는 Recall/F1/ROC-AUC.

## 0. 설정 · 데이터 로드

In [1]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import (recall_score, precision_score, f1_score,
                             roc_auc_score, precision_recall_curve, classification_report)

PROC_DIR = os.path.join('..', 'data', 'processed')
MODEL_DIR = os.path.join('..', 'outputs', 'models')
RANDOM_STATE = 42

data = np.load(os.path.join(PROC_DIR, 'secom_processed.npz'), allow_pickle=True)
X_clean, y_clean = data['X_train_clean'], data['y_train_clean']  # SMOTE 전 (튜닝/CV용)
X_test, y_test = data['X_test'], data['y_test']
print(f'튜닝용 train(clean): {X_clean.shape}, 불량 {int(y_clean.sum())}개')
print(f'test: {X_test.shape}, 불량 {int(y_test.sum())}개')

튜닝용 train(clean): (1253, 450), 불량 83개
test: (314, 450), 불량 21개


In [2]:
# 공통 유틸: (1) F1 최대 임계값 선택, (2) test 지표 계산
skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)

def pick_threshold(pipe, X, y):
    # 누수 없는 OOF 확률로 F1 최대 임계값 선택
    oof = cross_val_predict(pipe, X, y, cv=skf, method='predict_proba', n_jobs=-1)[:, 1]
    prec, rec, thr = precision_recall_curve(y, oof)
    f1s = 2 * prec * rec / (prec + rec + 1e-12)
    return float(thr[int(np.argmax(f1s[:-1]))])

def eval_test(name, pipe, threshold):
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= threshold).astype(int)
    return {'model': name, 'threshold': round(threshold, 3),
            'Recall': round(recall_score(y_test, pred), 3),
            'Precision': round(precision_score(y_test, pred, zero_division=0), 3),
            'F1': round(f1_score(y_test, pred, zero_division=0), 3),
            'ROC-AUC': round(roc_auc_score(y_test, proba), 3)}

## 1. RandomForest 튜닝

탐색 공간: 트리 수·깊이·분할 조건·피처 샘플링·클래스 가중치. 파라미터 이름은 파이프라인 단계명(`clf__`)을 접두로 붙인다.

In [3]:
# RF 파이프라인 + 탐색 공간
rf_pipe = ImbPipeline([('smote', SMOTE(random_state=RANDOM_STATE)),
                       ('clf', RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE))])
rf_space = {
    'clf__n_estimators': [200, 300, 500],
    'clf__max_depth': [None, 10, 20, 30],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__max_features': ['sqrt', 'log2', 0.3],
    'clf__class_weight': [None, 'balanced'],
}
rf_search = RandomizedSearchCV(rf_pipe, rf_space, n_iter=20, scoring='f1',
                               cv=skf, random_state=RANDOM_STATE, n_jobs=-1, verbose=0)
rf_search.fit(X_clean, y_clean)
print('RF 최적 파라미터:')
for k, v in rf_search.best_params_.items():
    print(f'  {k} = {v}')
print(f'RF CV F1(best): {rf_search.best_score_:.3f}')

/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_d

/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


...

/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


RF 최적 파라미터:
  clf__n_estimators = 300
  clf__min_samples_split = 5
  clf__min_samples_leaf = 1
  clf__max_features = 0.3
  clf__max_depth = 10
  clf__class_weight = balanced
RF CV F1(best): 0.120


In [4]:
# 튜닝된 RF: 임계값 선택 후 test 평가
rf_best = rf_search.best_estimator_
rf_thr = pick_threshold(rf_best, X_clean, y_clean)
rf_tuned_metrics = eval_test('RandomForest(튜닝)', rf_best, rf_thr)
print(rf_tuned_metrics)

/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_d

{'model': 'RandomForest(튜닝)', 'threshold': 0.279, 'Recall': 0.571, 'Precision': 0.203, 'F1': 0.3, 'ROC-AUC': np.float64(0.768)}


## 2. XGBoost 튜닝

탐색 공간: 트리 수·깊이·학습률·샘플링·자식 가중치·불량 가중치(scale_pos_weight).

In [5]:
# XGB 파이프라인 + 탐색 공간
xgb_pipe = ImbPipeline([('smote', SMOTE(random_state=RANDOM_STATE)),
                        ('clf', XGBClassifier(eval_metric='auc', random_state=RANDOM_STATE, n_jobs=-1))])
xgb_space = {
    'clf__n_estimators': [200, 300, 500],
    'clf__max_depth': [3, 4, 6],
    'clf__learning_rate': [0.01, 0.05, 0.1],
    'clf__subsample': [0.7, 0.8, 1.0],
    'clf__colsample_bytree': [0.7, 0.8, 1.0],
    'clf__min_child_weight': [1, 3, 5],
}
xgb_search = RandomizedSearchCV(xgb_pipe, xgb_space, n_iter=20, scoring='f1',
                                cv=skf, random_state=RANDOM_STATE, n_jobs=-1, verbose=0)
xgb_search.fit(X_clean, y_clean)
print('XGB 최적 파라미터:')
for k, v in xgb_search.best_params_.items():
    print(f'  {k} = {v}')
print(f'XGB CV F1(best): {xgb_search.best_score_:.3f}')

/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_d

/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_d

/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_d

/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_d

...

/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_d

/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


XGB 최적 파라미터:
  clf__subsample = 0.7
  clf__n_estimators = 200
  clf__min_child_weight = 1
  clf__max_depth = 3
  clf__learning_rate = 0.05
  clf__colsample_bytree = 0.7
XGB CV F1(best): 0.201


In [6]:
# 튜닝된 XGB: 임계값 선택 후 test 평가
xgb_best = xgb_search.best_estimator_
xgb_thr = pick_threshold(xgb_best, X_clean, y_clean)
xgb_tuned_metrics = eval_test('XGBoost(튜닝)', xgb_best, xgb_thr)
print(xgb_tuned_metrics)

/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/kimmilim/Desktop/미림/2026-1/SK하이닉스_중반기/반도체플젝/.venv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_d

{'model': 'XGBoost(튜닝)', 'threshold': 0.345, 'Recall': 0.286, 'Precision': 0.375, 'F1': 0.324, 'ROC-AUC': np.float64(0.734)}


## 3. 기본값 vs 튜닝 비교

Step 3에서 저장한 기본 모델의 test 성능과 튜닝 결과를 같은 표에서 비교한다.

In [7]:
# Step 3 기본 모델(저장본)으로 baseline test 지표 재계산
def eval_saved(name, path):
    b = joblib.load(path)
    m, thr = b['model'], b['threshold']
    proba = m.predict_proba(X_test)[:, 1]
    pred = (proba >= thr).astype(int)
    return {'model': name, 'threshold': round(thr, 3),
            'Recall': round(recall_score(y_test, pred), 3),
            'Precision': round(precision_score(y_test, pred, zero_division=0), 3),
            'F1': round(f1_score(y_test, pred, zero_division=0), 3),
            'ROC-AUC': round(roc_auc_score(y_test, proba), 3)}

rows = [
    eval_saved('RandomForest(기본)', os.path.join(MODEL_DIR, 'random_forest.joblib')),
    rf_tuned_metrics,
    eval_saved('XGBoost(기본)', os.path.join(MODEL_DIR, 'xgboost.joblib')),
    xgb_tuned_metrics,
]
compare = pd.DataFrame(rows).set_index('model')[['threshold', 'Recall', 'Precision', 'F1', 'ROC-AUC']]
print('기본값 vs 튜닝 비교 (test 기준)')
print(compare.to_string())

기본값 vs 튜닝 비교 (test 기준)
                  threshold  Recall  Precision     F1  ROC-AUC
model                                                         
RandomForest(기본)      0.320   0.333      0.292  0.311    0.805
RandomForest(튜닝)      0.279   0.571      0.203  0.300    0.768
XGBoost(기본)           0.116   0.333      0.241  0.280    0.724
XGBoost(튜닝)           0.345   0.286      0.375  0.324    0.734


In [8]:
# 튜닝 모델 저장 (Step 7 최종 비교에서 재사용)
joblib.dump({'model': rf_best, 'threshold': rf_thr, 'params': rf_search.best_params_},
            os.path.join(MODEL_DIR, 'random_forest_tuned.joblib'))
joblib.dump({'model': xgb_best, 'threshold': xgb_thr, 'params': xgb_search.best_params_},
            os.path.join(MODEL_DIR, 'xgboost_tuned.joblib'))
print('튜닝 모델 저장 완료 → outputs/models/{random_forest_tuned, xgboost_tuned}.joblib')

튜닝 모델 저장 완료 → outputs/models/{random_forest_tuned, xgboost_tuned}.joblib


### 포트폴리오 인사이트 (Step 6)

- **"최적화는 해봤나?"에 대한 답**: 손으로 정한 기본값에 머무르지 않고 RandomizedSearchCV로 탐색 공간을 정의해 체계적으로 최적화했다.
- **누수 없는 튜닝**: SMOTE를 CV 파이프라인 안에 두어 탐색·임계값 선택을 모두 train 안에서 수행했다. test는 최종 평가에만 사용 → 정직한 비교.
- **현실적 해석**: SECOM은 신호가 약해 튜닝으로 성능이 폭등하진 않는다. 중요한 건 *수치 점프*가 아니라 *과적합 없이 한계를 확인하는 과정*이며, 이 자체가 방법론적 신뢰도를 보여준다.

**→ 다음 단계: `07_lightgbm_comparison.ipynb` (LightGBM을 동일 방식으로 추가해 3-모델 최종 비교)**